# Matching

This notebook is based on this chapter: https://matheusfacure.github.io/python-causality-handbook/10-Matching.html



In [1]:
import pandas as pd
import os

from tabulate import tabulate
import statsmodels.formula.api as smf

In [2]:
ROOTDIR = os.getcwd()
DATADIR = os.path.join(ROOTDIR, 'data')

Suppose there is a drug that is meant to reduce the length of an illness.
The drug has a different (heterogeneous) effects on men than it does on women: it reduces the length of the illness in men by 3 days, and by 2 days in women. 

There are ten participants in a study of this drug: 6 men, and 4 women. 
The number of days they are ill are the observation of interest. 


In [3]:
drug_df = pd.DataFrame(dict(
    sex= ["M","M","M","M","M","M", "W","W","W","W"],
    treatment=[1,1,1,1,1,0,  1,0,1,0],
    days=[5,5,5,5,5,8,  2,4,2,4]
))

First, try the naive estimate of the treatment effect: subtract the mean outcome for the control group
from the mean outcome for the treated group:

In [4]:
# alt
drug_df.groupby('treatment')['days'].mean().diff()
# more explicit
(
    drug_df.loc[drug_df['treatment'] == 1, 'days'].mean()
    - drug_df.loc[drug_df['treatment'] == 0, 'days'].mean()
)

np.float64(-1.1904761904761898)

The the naive treatment effect is smaller than the known effect (-3 for men, and -2 for women).
This is because men get sicker from this illness, and more men receive the drug.

We can calculate the true average treatment effect by weighing taking a weighted average of the true effect 
for men and the true effect for women.
\begin{equation}
ATE = \frac{(-3 \cdot 6) + (-2 \cdot 4)}{10} = -2.6
\end{equation}
If you only had the data, you can estimate this ATE using the code below:

In [5]:
row_m = (drug_df['sex'] == 'M')
row_w = (drug_df['sex'] == 'W')
row_treat = (drug_df['treatment'] == 1)
row_control = (drug_df['treatment'] == 0)

drug_df.loc[(row_m & row_treat), 'days'].mean()

ate_m = (
    drug_df.loc[(row_m & row_treat), 'days'].mean() 
    - drug_df.loc[(row_m & row_control), 'days'].mean() 
)

ate_w = (
    drug_df.loc[(row_w & row_treat), 'days'].mean() 
    - drug_df.loc[(row_w & row_control), 'days'].mean() 
)

weight_m = row_m.sum() / len(drug_df)
weight_w = row_w.sum() / len(drug_df)

ate = ate_m * weight_m + ate_w * weight_w
print(f'ATE: {ate:.3f}')

ATE: -2.600


This is calculated by partitioning the data by sex, and calculating the effect for each partition, and weighing the effects
by overall prevalence in the data. 

Notice you get a similar effect by running a regression on this data.

In [6]:
ols_mod = smf.ols('days ~ treatment + C(sex)', data=drug_df)
ols_fit = ols_mod.fit()
ols_fit.summary(slim=True)

/opt/anaconda3/envs/venv_econ726/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=10 observations were given.
  return hypotest_fun_in(*args, **kwds)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   days   R-squared:                       0.983
Model:                            OLS   Adj. R-squared:                  0.978
No. Observations:                  10   F-statistic:                     200.5
Covariance Type:            nonrobust   Prob (F-statistic):           6.61e-07
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       7.5455      0.188     40.093      0.000       7.100       7.990
C(sex)[T.W]    -3.3182      0.176    -18.849      0.000      -3.734      -2.902
treatment      -2.4545      0.188    -13.042      0.000      -2.900      -2.010
===============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

The regression estimator is also splitting the data into cells based on sex and 
calculating the effects for each cell (in the parlance of MHE, calculating each $\delta_X$).
Regression and matching differ when it comes to the weights used to combine these effects into a single weighted average effect.
* Matching uses distribution of covariates to weigh effects
* Regression uses weights that are proportional to the variance of the treatment in that group.

Note that the variance of the treatment in this example is higher for women than for men (this follows because there is only one man in the control group).
Because the regression places a slightly higher weight on women, the estimator is a bit closer to -2 than the ATE.

Mean and variance of treatment for men:
\begin{align}
\mu =& \frac{1}{6} (1 + 1 + 1 + 1 + 1 + 0) = \frac{5}{6} \\
\text{Var} (D_i \vert \text{sex} = M) =& \frac{1}{6} \sum_{i=1}^6 \left(D_i - \frac{5}{6}\right)^2 
= \frac{1}{6} \left( \left(1 - \frac{5}{6}\right)^2 \cdot 5 + \left(0 - \frac{5}{6}\right)^2 \right) \\
=& \frac{1}{6} \left( \left(\frac{1}{6}\right)^2 \cdot 5 + \left(-\frac{5}{6}\right)^2 \right) 
= \frac{1}{6} \left( \frac{1}{36} \cdot 5 + \frac{25}{36} \right)
= \frac{1}{6} \left( \frac{30}{36} \right) \\
=& \frac{5}{36}
\end{align}
Notice you can just calculate this using code:

In [7]:
# notice you can calculate the mean and variance using code
# here i adjust ddof for the population estimate
var_men = drug_df['treatment'].loc[drug_df['sex'] == 'M'].var(ddof=0)
print(f'Variance for men: {var_men:.3f}')

var_women = drug_df['treatment'].loc[drug_df['sex'] == 'W'].var(ddof=0)
print(f'Variance for women: {var_women:.3f}')

Variance for men: 0.139
Variance for women: 0.250


## Subclassification Estimator

In [9]:
# csv_file = os.path.join(DATADIR, 'trainees.csv')
url = "https://raw.githubusercontent.com/matheusfacure/python-causality-handbook/master/causal-inference-for-the-brave-and-true/data/trainees.csv"
df = pd.read_csv(url)

In [10]:
# naive comparison of trainees vs non trainees
mean_df = df.groupby(['trainees'])['earnings'].mean()

print(tabulate(mean_df.to_frame(), headers='keys'))

naive_diff = mean_df.loc[1] - mean_df.loc[0]

print(f'Naive difference: {naive_diff:.2f}')

  trainees    earnings
----------  ----------
         0     20723.8
         1     16426.3
Naive difference: -4297.49


In [11]:
# problem: trainees are younger

df.groupby('trainees')['age'].describe()

,count,mean,std,min,25%,50%,75%,max
trainees,,,,,,,,
0,21.0,33.000000,8.977750,23.0,26.0,31.0,35.0,54.0
1,19.0,28.473684,4.526078,23.0,25.5,28.0,29.0,43.0


In [12]:
df.loc[df['trainees'] == 0]

,unit,trainees,age,earnings
19,20,0,43,20900
20,21,0,50,31000
21,22,0,30,21000
22,23,0,27,9300
23,24,0,54,41100
24,25,0,48,29800
25,26,0,39,42000
26,27,0,28,8800
27,28,0,24,25500
28,29,0,33,15500


In [13]:
# potential matches
df.loc[
    ((df['unit'] == 1) & (df['trainees'] == 1))
    | ((df['unit'] == 27))
]

# find the age of unit 2
df.loc[df['unit'] == 2, 'age']
df.loc[df['age'] == 34]
# match unit 2 with unit 34

df.loc[df['unit'] == 3, 'age']
df.loc[df['age'] == 29]
# pair with unit 37

df.loc[df['unit'] == 5, 'age']
df.loc[df['age'] == 29]
# use unit 37 again

# if there are multiple matches, pick randomly

,unit,trainees,age,earnings
2,3,1,29,14400
4,5,1,29,6100
12,13,1,29,12500
35,37,0,29,6200


In [14]:
df_control = df.loc[df['trainees'] == 0]

# keep non-duplicated results by age in control group
df_control = df_control.loc[~df_control['age'].duplicated()]

df_control
df.loc[df['trainees'] == 1]
matches = pd.merge(
    left=df.loc[df['trainees'] == 1],
    right=df_control,
    how='left',
    on='age',
    validate='m:1',
    suffixes=['_treatment', '_control']
)
matches['te'] = matches['earnings_treatment'] - matches['earnings_control']
matches.head()

,unit_treatment,trainees_treatment,age,earnings_treatment,unit_control,trainees_control,earnings_control,te
0,1,1,28,17700,27,0,8800,8900
1,2,1,34,10200,34,0,24200,-14000
2,3,1,29,14400,37,0,6200,8200
3,4,1,25,20800,35,0,23300,-2500
4,5,1,29,6100,37,0,6200,-100


In [15]:
ate_ot = matches['te'].mean()
print(f'ATE: {ate_ot:.2f}')

# note: because this data is in a long format, you can just take the mean 
# to get the ATE on treated. Notice the result is the same if you
# explicitly calculate the weights

matches.groupby('age')['te'].mean()

prob_x_treated = matches.groupby('age')['te'].count() / len(matches)

(matches.groupby('age')['te'].mean() * prob_x_treated).sum()


ATE: 2457.89


np.float64(2457.8947368421054)

In [16]:
# to get the ATE, you need the wieghts from the overall distribution
weights = df.groupby('age')['unit'].count() / len(df)

ate_df = pd.merge(
    left=matches.groupby('age')['te'].mean(),
    right=weights,
    on='age',
    how='left'
)
(ate_df['te'] * ate_df['unit']).sum()

np.float64(2385.833333333334)